# 0-Tokenizer

Tokenizer 把自然语言切成 token 再映射成 id，是模型和文本之间的词典。MiniMind-3 官方词表已经放在 `../model/`（`tokenizer.json` + `tokenizer_config.json`），词表大小 `6400`，BPE + ByteLevel。

**不建议重新训练官方词表**：换词表后权重、数据、推理接口都会对不上。`trainer/train_tokenizer.py` 和下面的代码只供学习。


## 子词分词算法

常见三种：

1. **BPE**：迭代合并出现最频繁的符号对，直到达到目标词表大小。MiniMind 用的就是它。
2. **WordPiece**：同样做合并，但选择让训练数据似然最大的那一对。
3. **Unigram**：从很大的候选词表往下删，用对数似然决定去留。

MiniMind 选短词表，是为了压住 embedding / lm_head 在小模型里的参数占比。


## 先看官方词表

主线特殊标记已经对齐 ChatML / Qwen 风格：`<|im_start|>`、`<|im_end|>`、`<think>`、`<tool_call>`，并预留了 buffer token。


In [ ]:
import os, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
print("cwd:", os.getcwd())

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("../model")
print("vocab_size:", tok.vocab_size)
print("bos/eos/pad/unk:", tok.bos_token, tok.eos_token, tok.pad_token, tok.unk_token)
print("sample specials:", [t for t in ["<|im_start|>", "<|im_end|>", "<think>", "</think>", "<tool_call>", "</tool_call>"] if t in tok.get_vocab()])
print(tok.apply_chat_template([
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！"}
], tokenize=False))


## 自己训一个最小 BPE（演示）

下面用 `toydata/tokenizer_data.jsonl` 训一个玩具词表，写到 `./model/toy_tokenizer/`，**不会覆盖** `../model/`。流程与 `trainer/train_tokenizer.py` 相同，只是词表更小、特殊标记更少。


In [ ]:
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

special_tokens = [
    "<|endoftext|>", "<|im_start|>", "<|im_end|>",
    "<think>", "</think>", "<tool_call>", "</tool_call>",
    "<tool_response>", "</tool_response>",
]
trainer = trainers.BpeTrainer(
    vocab_size=512,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens=special_tokens,
)


In [ ]:
import json

def read_texts(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            yield obj.get("text") or json.dumps(obj, ensure_ascii=False)

tokenizer.train_from_iterator(read_texts("./toydata/tokenizer_data.jsonl"), trainer=trainer)
tokenizer.decoder = decoders.ByteLevel()
print("id(<|im_start|>) =", tokenizer.token_to_id("<|im_start|>"))
print("id(<|im_end|>) =", tokenizer.token_to_id("<|im_end|>"))


In [ ]:
import os, json
from transformers import AutoTokenizer

out_dir = "./model/toy_tokenizer"
os.makedirs(out_dir, exist_ok=True)
tokenizer.save(os.path.join(out_dir, "tokenizer.json"))

config = {
    "add_bos_token": False,
    "add_eos_token": False,
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
    "model_max_length": 32768,
    "tokenizer_class": "PreTrainedTokenizerFast",
    "chat_template": "{% for message in messages %}{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}",
}
with open(os.path.join(out_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

toy = AutoTokenizer.from_pretrained(out_dir)
print(toy.apply_chat_template([{"role": "user", "content": "你好"}], tokenize=False, add_generation_prompt=True))
print("toy vocab:", toy.vocab_size)


完整官方 chat_template 还支持 `tools`、`open_thinking`、`<tool_call>` / `<think>`，见 `../model/tokenizer_config.json`。后面的 notebook 一律加载官方 `../model`，不要用这个玩具词表训主线模型。
